# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents the observed daily performance of one content item for one client on one report date.

Time window: For development and verification, I use March 2026 as the mid-panel month. The warehouse covers historical daily performance, but available history can differ by client, so March is used as the working window rather than assuming every client has the same history.

The prediction/analysis target is the observed decline label used in W02: is_declining_label, derived from trend_direction. This is treated as a label/proxy, not as a feature.



In [14]:


from google.colab import userdata
import duckdb

# Get token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()

# Create DuckDB Hugging Face secret
con.execute(f"""
    CREATE SECRET (
        TYPE huggingface,
        TOKEN '{hf_token}'
    )
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

count = con.sql(f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
""")

print(count)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘



In [2]:
query_1 = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_unavailable_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= '2026-03-01'
  AND report_date < '2026-04-01'
"""

con.sql(query_1)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬─────────────────┬─────────────────┬────────────────────┬──────────────────────┐
│ row_count │ min_report_date │ max_report_date │ ga4_available_rows │ ga4_unavailable_rows │
│   int64   │      date       │      date       │       int64        │        int64         │
├───────────┼─────────────────┼─────────────────┼────────────────────┼──────────────────────┤
│   9841378 │ 2026-03-01      │ 2026-03-31      │             413966 │              9427412 │
└───────────┴─────────────────┴─────────────────┴────────────────────┴──────────────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

FEATURE FIELDS The candidate features are signals that are available from the performance data without using the decline outcome itself: gsc_clicks gsc impressions gsc_ctr gsc_avg_position ga4_engagement_rate

The label and the fields used to construct it must not bt used as model features

CONTEXT content_id - identifies the content item client_id - identifies the client and is useful for grouping/splitting report_date- identifies the observation date content_type- provides content context ga4_data_available - indicates whether GA4 data is available for the row

these fields provide identity, grouping, timing or availability information rather thaan model learning signals

EXCLUDED trend_direction -excluded because it directly contributes to the label trend_pct - excluded because it is upstream of trend_direction and therefore directly connected to the label content_id - excluded from model learning because it is an identifier client_id - excluded from model learning because it is an identifier and could cause client specific memorization provider_used- excluded because it is not a content performance signal model_used - excluded because it describes the system/model rather than the content signal

**impressions_last_30d and impressions_prev_30d **- excluded because their time window can overlap the outcome window and therefore creates a leakage risk

The deliberate exclusion is future /outcome derived information: fields used to construct the decline label are never treated as features

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect the columns in the table
schema = con.sql("""
DESCRIBE
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
""")
schema

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [5]:
query_2 = """
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= '2026-03-01'
  AND report_date < '2026-04-01'
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
"""

con.sql(query_2)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────────┐
│ client_hash_id │ content_hash_id │ report_date │ row_count │
│    varchar     │     varchar     │    date     │   int64   │
├────────────────┴─────────────────┴─────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

In [6]:
query_3 = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= '2026-03-01'
  AND report_date < '2026-04-01'
"""

con.sql(query_3)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.

DATA LIMITS: This dataset does not provide equally completer history for every client or content item, so observations should not be treated s if they have the same historical coverage some early observations may have GSC data available while GA4 data unavailable , so GA4 based features can have missing values for valid observations. The data also contains time windowed fields, so features must be checked carefullyto ensure their measurement window does not overlap the period used to define the decline outcome. Because of these limits, this dataset can support measured, directional analysis of observed search and engagement performance, but it cannot gurantee that every content item has the same data coverage or that missing GA4 data means zero engagement


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.